# 33. HS4–HS6 Probability Fusion

32번과 동일한 195개 평가 표본, 11개 HS6 클래스, 29개 제품군 및 LOPO Fold를 유지합니다. HS4 확률과 HS4 내부 HS6 조건부 확률을 결합하고, 기존 평면 분류와 비교합니다. Oracle은 진단용 상한선으로만 사용합니다. Google Drive는 사용하지 않습니다.

## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy
import json, requests, re, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/33_probability_fusion_results')
WORK.mkdir(parents=True,exist_ok=True)
print(WORK)

## 1. 29번과 동일한 데이터 구성

In [ ]:
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print(len(eval_df),len(labels),eval_df.product_family.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29

## 2. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_33.npz'
if CACHE.exists():
 z=np.load(CACHE,allow_pickle=True); query_emb,ruling_emb=z['query_emb'],z['ruling_emb']
else:
 sbert=SentenceTransformer('intfloat/multilingual-e5-base')
 query_emb=sbert.encode(['query: '+str(x) for x in eval_df.query_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
 ruling_emb=sbert.encode(['passage: '+str(x) for x in eval_df.ruling_text],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
 np.savez_compressed(CACHE,query_emb=query_emb,ruling_emb=ruling_emb)
print(query_emb.shape,ruling_emb.shape)

## 3. 점수 집계 및 확률 결합

각 코드별 학습 사례 유사도 상위 3개 평균을 사용합니다. 확률은 Fold 내부 학습 자료의 점수만으로 계산합니다. 온도 조합은 사전에 고정한 민감도 분석이며, 최종 모델 선택을 위한 외부 테스트 튜닝으로 해석하지 않습니다.

In [ ]:
def aggregate_code_scores(scores,codes,top_n=3):
 d=pd.DataFrame({'code':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
 return d.groupby('code').score.apply(lambda x:x.nlargest(min(top_n,len(x))).mean()).sort_values(ascending=False)
def softmax(x,temp):
 x=np.asarray(x,dtype=float); z=(x-x.max())/temp; e=np.exp(z); return e/e.sum()
def fuse_hierarchical(hs4_scores,hs6_scores,t4,t6):
 p4=pd.Series(softmax(hs4_scores.values,t4),index=hs4_scores.index)
 out={}
 for h4,group in hs6_scores.groupby(hs6_scores.index.to_series().str[:4]):
  if h4 not in p4.index: continue
  cond=softmax(group.values,t6)
  for code,p in zip(group.index,cond): out[str(code)]=float(p4.loc[h4]*p)
 return pd.Series(out).sort_values(ascending=False)
def result_path(fold): return WORK/f'fold_{fold:02d}.csv'
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs6,groups=eval_df.product_family))
print('Fold:',len(splits))

## 4. Fold별 실행

각 Fold의 홀드아웃 제품군은 학습 및 점수 집계에서 제외됩니다. 완료된 Fold 파일은 자동으로 건너뜁니다.

In [ ]:
FOLD_START=1
FOLD_END=5
TEMPERATURES=[0.05,0.10,0.20]
for fold,(tr,te) in enumerate(splits,1):
 if not FOLD_START<=fold<=FOLD_END: continue
 if result_path(fold).exists(): print('Fold',fold,'완료 - skip'); continue
 train=eval_df.iloc[tr].copy(); train_hs6=train.hs6.astype(str).to_numpy(); train_hs4=train.hs4.astype(str).to_numpy()
 if not set(eval_df.iloc[te].hs6.astype(str)).issubset(set(train_hs6)): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS6')
 rows=[]
 for global_i in te:
  scores=ruling_emb[tr]@query_emb[global_i]
  hs4_scores=aggregate_code_scores(scores,train_hs4); hs6_scores=aggregate_code_scores(scores,train_hs6)
  flat=hs6_scores.sort_values(ascending=False)
  true4=str(eval_df.iloc[global_i].hs4); true6=str(eval_df.iloc[global_i].hs6)
  rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':true4,'true_hs6':true6,'model':'Flat_SBERT_ruling','predicted_hs6':str(flat.index[0]),'top3_candidates':'|'.join(map(str,flat.index[:3])),'predicted_hs4':str(flat.index[0])[:4]})
  oracle=hs6_scores[[str(c)[:4]==true4 for c in hs6_scores.index]].sort_values(ascending=False)
  rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':true4,'true_hs6':true6,'model':'Oracle_HS4_diagnostic','predicted_hs6':str(oracle.index[0]),'top3_candidates':'|'.join(map(str,oracle.index[:3])),'predicted_hs4':true4})
  for t4 in TEMPERATURES:
   for t6 in TEMPERATURES:
    fused=fuse_hierarchical(hs4_scores,hs6_scores,t4,t6)
    rows.append({'row_index':int(global_i),'fold':fold,'heldout_family':eval_df.iloc[global_i].product_family,'true_hs4':true4,'true_hs6':true6,'model':f'Probability_Fusion_T4_{t4:.2f}_T6_{t6:.2f}','predicted_hs6':str(fused.index[0]),'top3_candidates':'|'.join(map(str,fused.index[:3])),'predicted_hs4':str(fused.index[0])[:4]})
 pd.DataFrame(rows).to_csv(result_path(fold),index=False,encoding='utf-8-sig')
 print(f'Fold {fold}/{len(splits)} 저장 완료 - {len(te)}건')
print('실행 구간 완료')

## 5. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_*.csv')); missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)}'); print('누락:',missing)
BACKUP='/content/33_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
 for p in WORK.glob('fold_*.csv'): z.write(p,p.name)
print('백업 ZIP:',BACKUP)
from google.colab import files
files.download(BACKUP)

## 6. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
 if name.endswith('.zip'):
  with zipfile.ZipFile(name) as z: z.extractall(WORK)
  print('복구 완료:',name)

## 7. 전체 통합 및 평가

In [ ]:
files_list=sorted(WORK.glob('fold_*.csv'))
done=sorted(int(p.stem.split('_')[1]) for p in files_list); missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
if missing: raise ValueError('미완료 Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any(): raise ValueError('중복 예측 존재')
def evaluate(g):
 yt=g.true_hs6.astype(str); yp=g.predicted_hs6.astype(str); top=g.top3_candidates.fillna('').str.split('|')
 return pd.Series({'n':len(g),'HS4_accuracy':np.mean(yt.str[:4].to_numpy()==yp.str[:4].to_numpy()),'HS6_accuracy':accuracy_score(yt,yp),'Macro_F1':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),'Weighted_F1':f1_score(yt,yp,average='weighted',zero_division=0),'Top3_accuracy':np.mean([t in c for t,c in zip(yt,top)])})
summary=pred.groupby('model').apply(evaluate,include_groups=False).reset_index().sort_values('Macro_F1',ascending=False)
class_rows=[]
for model,g in pred.groupby('model'):
 p,r,f,s=precision_recall_fscore_support(g.true_hs6.astype(str),g.predicted_hs6.astype(str),labels=labels,zero_division=0)
 for c,pv,rv,fv,sv in zip(labels,p,r,f,s): class_rows.append({'model':model,'HS6':c,'support':int(sv),'precision':pv,'recall':rv,'F1':fv})
class_metrics=pd.DataFrame(class_rows)
display(summary.round(4)); display(class_metrics.sort_values(['model','F1']).round(4))
OUT=WORK/'final'; OUT.mkdir(exist_ok=True)
pred.to_csv(OUT/'33_oof_predictions.csv',index=False,encoding='utf-8-sig'); summary.to_csv(OUT/'33_model_comparison.csv',index=False,encoding='utf-8-sig'); class_metrics.to_csv(OUT/'33_class_metrics.csv',index=False,encoding='utf-8-sig')
for model,g in pred.groupby('model'):
 cm=pd.DataFrame(confusion_matrix(g.true_hs6.astype(str),g.predicted_hs6.astype(str),labels=labels),index=labels,columns=labels)
 cm.to_csv(OUT/f'confusion_{model}.csv',encoding='utf-8-sig')
ZIP='/content/33_probability_fusion_results.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as z:
 for p in OUT.glob('*.csv'): z.write(p,p.name)
print('최종 ZIP:',ZIP)

## 8. 최종 결과 PC 다운로드

In [ ]:
from google.colab import files
files.download('/content/33_probability_fusion_results.zip')